# Multi-agent avec CrewAI

CrewAI permet de faire collaborer plusieurs agents IA, chacun avec un **role**, un **objectif**, et eventuellement un **modele different**.

Dans cet exemple, pipeline a 3 etages :
- **Agent Chercheur** (Groq, tres rapide) : rassemble des faits bruts sur un sujet.
- **Agent Redacteur** (Claude, plus rigoureux) : reprend ces faits et redige une reponse structuree.
- **Agent Relecteur** (Gemini) : relit le texte du redacteur, corrige les eventuelles erreurs et le peaufine pour la sortie finale.

Les trois agents travaillent en **sequence** : chaque resultat sert d'entree au suivant (pipeline d'agents specialises).

Note sur l'installation :
- **Anthropic** et **Gemini** sont des fournisseurs "natifs" de CrewAI, mais chacun necessite son propre extra pip (`crewai[anthropic]`, `crewai[google-genai]`).
- **Groq** n'est pas natif du tout : CrewAI passe par **LiteLLM** pour lui, d'ou l'installation du package `litellm` en plus.

In [ ]:
!pip install -q "crewai[anthropic,google-genai]" litellm

## Configuration des cles API

Reutilise les memes cles que dans `test_gemini_groq.ipynb` (Secrets Colab `ANTHROPIC_API_KEY`, `GROQ_API_KEY`, `GEMINI_API_KEY`, sinon saisie manuelle).

In [ ]:
import os
from getpass import getpass

def get_key(env_name, prompt):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get(env_name)
    except Exception:
        pass
    if not key:
        key = getpass(prompt)
    os.environ[env_name] = key
    return key

anthropic_key = get_key('ANTHROPIC_API_KEY', 'Entre ta cle API Anthropic: ')
groq_key = get_key('GROQ_API_KEY', 'Entre ta cle API Groq: ')
gemini_key = get_key('GEMINI_API_KEY', 'Entre ta cle API Gemini: ')

## Definir les modeles (un par agent)

CrewAI utilise la convention LiteLLM : `"<fournisseur>/<nom_du_modele>"`.

In [ ]:
from crewai import LLM

groq_llm = LLM(
    model='groq/llama-3.1-8b-instant',
    api_key=groq_key,
)

claude_llm = LLM(
    model='anthropic/claude-haiku-4-5',
    api_key=anthropic_key,
)

gemini_llm = LLM(
    model='gemini/gemini-2.5-flash',
    api_key=gemini_key,
)

## Correctif pour un bug connu de CrewAI (Groq)

CrewAI insere un champ `cache_breakpoint` dans les messages systeme, prevu pour la mise en cache Anthropic, mais l'envoie aussi aux autres fournisseurs (Groq, OpenAI-compatible) qui le rejettent avec une erreur `BadRequestError`. C'est un bug connu ([issue #5886](https://github.com/crewAIInc/crewAI/issues/5886)), pas encore corrige dans le package. On neutralise la fonction fautive avec un monkey-patch avant de creer les agents.

In [ ]:
import crewai.llms.cache as _crewai_cache

# Neutralise l'injection du champ 'cache_breakpoint' dans les messages,
# qui fait planter les fournisseurs autres qu'Anthropic (ex: Groq).
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

## Definir les agents et les taches

In [ ]:
from crewai import Agent, Task, Crew, Process

chercheur = Agent(
    role='Chercheur',
    goal="Rassembler rapidement des faits precis et pertinents sur le sujet demande",
    backstory="Tu es un chercheur efficace qui produit des listes de faits bruts, sans fioritures.",
    llm=groq_llm,
    verbose=True,
)

redacteur = Agent(
    role='Redacteur',
    goal="Transformer des faits bruts en une reponse claire, structuree et bien ecrite",
    backstory="Tu es un redacteur rigoureux qui organise l'information de facon pedagogique.",
    llm=claude_llm,
    verbose=True,
)

relecteur = Agent(
    role='Relecteur',
    goal="Relire un texte, corriger les erreurs et l'ameliorer sans en changer le sens",
    backstory="Tu es un relecteur exigeant qui verifie la clarte, la coherence et l'orthographe.",
    llm=gemini_llm,
    verbose=True,
)

sujet = "les avantages et inconvenients de l'energie solaire pour une maison individuelle en France"

tache_recherche = Task(
    description=f"Liste 5 a 8 faits factuels et verifiables sur : {sujet}. Format : liste a puces, une phrase par fait.",
    expected_output="Une liste a puces de faits bruts.",
    agent=chercheur,
)

tache_redaction = Task(
    description=(
        "A partir des faits fournis par le chercheur, redige une reponse structuree en 3 parties : "
        "avantages, inconvenients, et une recommandation finale en une phrase."
    ),
    expected_output="Une reponse structuree en 3 parties (avantages / inconvenients / recommandation).",
    agent=redacteur,
    context=[tache_recherche],
)

tache_relecture = Task(
    description=(
        "Relis le texte redige. Corrige toute erreur factuelle, de grammaire ou de clarte, "
        "sans changer la structure en 3 parties. Produis la version finale, prete a etre publiee."
    ),
    expected_output="La version finale corrigee et peaufinee du texte, en 3 parties.",
    agent=relecteur,
    context=[tache_redaction],
)

## Lancer la crew (execution sequentielle)

In [ ]:
crew = Crew(
    agents=[chercheur, redacteur, relecteur],
    tasks=[tache_recherche, tache_redaction, tache_relecture],
    process=Process.sequential,
    verbose=True,
)

# Colab/Jupyter fait deja tourner une boucle asyncio : on utilise donc la
# version async de kickoff (via top-level await, supporte par Colab)
# plutot que crew.kickoff() qui provoque un RuntimeError dans ce contexte.
resultat = await crew.kickoff_async()

print('\n=== Reponse finale ===')
print(resultat)